# Ingestão Dinâmica de Dados para a Camada Bronze

## 1️⃣ Configuração do Ambiente

In [0]:
from pyspark.sql.utils import AnalysisException
import re
import unicodedata
import pandas as pd

## 2️⃣ Lista de arquivos para ingestão

In [0]:
tables = spark.catalog.listTables("dados_gov.imposto_de_renda_data_sources")

file_names = [t.name for t in tables]

print("Arquivos encontrados no Volume:")
print(file_names)

## 3️⃣ Funções auxiliares de normalização

In [0]:
def normalize_column(col_name):

    col_name = col_name.lower()

    col_name = unicodedata.normalize('NFKD', col_name)\
        .encode('ASCII','ignore')\
        .decode('utf-8')

    col_name = re.sub(r'[^a-zA-Z0-9_]', '_', col_name)
    col_name = re.sub('_+', '_', col_name)

    return col_name.strip('_')[:200]


def normalize_table_name(name):

    return name.replace("-", "_")

## 4️⃣ Configuração do Volume e Schema Bronze

In [0]:
catalog_name = "dados_gov"
schema_name = "bronze_layer"

# cria schema caso não exista
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {catalog_name}.{schema_name}")

print("Schema bronze_layer verificado")

## 5️⃣ Ingestão Dinâmica de Dados para Bronze

In [0]:
catalog_source = "dados_gov"
schema_source = "imposto_de_renda_data_sources"

catalog_name = "dados_gov"
schema_name = "bronze_layer"

for file_name in file_names:

    try:

        print("===================================")
        print(f"Processando dataset: {file_name}")

        source_table = f"{catalog_source}.{schema_source}.{file_name}"

        table_name = normalize_table_name(file_name)

        full_table_name = f"{catalog_name}.{schema_name}.delta_{table_name}"

        spark.sql(f"DROP TABLE IF EXISTS {full_table_name}")

        # ------------------------------------
        # LER TABELA SPARK
        # ------------------------------------

        if spark.catalog.tableExists(source_table):

            print(f"Lendo tabela: {source_table}")

            df_spark = spark.table(source_table)

            print("Tabela carregada com Spark")

            # ------------------------------------
            # CONVERTER PARA PANDAS
            # ------------------------------------

            df = df_spark.toPandas()

            print("Convertido para Pandas DataFrame")

            # ------------------------------------
            # VOLTAR PARA SPARK
            # ------------------------------------

            s_df = spark.createDataFrame(df)

            print("Convertido novamente para Spark DataFrame")

            # ------------------------------------
            # NORMALIZAR COLUNAS
            # ------------------------------------

            columns_df_bronze = [normalize_column(col) for col in s_df.columns]

            s_df = s_df.toDF(*columns_df_bronze)

            # ------------------------------------
            # SALVAR NA BRONZE
            # ------------------------------------

            print(f"Gravando tabela: {full_table_name}")

            s_df.write \
                .mode("overwrite") \
                .saveAsTable(full_table_name)

            print(f"Tabela criada: {full_table_name}")

        else:

            print(f"Tabela {source_table} não encontrada")

    except Exception as e:

        print(f"Erro ao processar {file_name}")
        print(str(e))